In [1]:
import larq as lq
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.python.client import device_lib
weight_quantizer = lq.quantizers.DoReFa(k_bit=6)
activation_quantizer = lq.quantizers.DoReFa(k_bit=6)

kwargs = dict(input_quantizer=activation_quantizer,
              kernel_quantizer=weight_quantizer,
              kernel_constraint="weight_clip")

def create_larq_model():
    model = models.Sequential()

    model.add(lq.layers.QuantConv2D(8, (3, 3),
                                    kernel_quantizer="ste_sign",
                                    kernel_constraint="weight_clip",
                                    use_bias=False,
                                    input_shape=(32, 32, 3)))
    model.add(layers.BatchNormalization(scale=False))
    model.add(layers.ReLU())

    # Layer 1
    model.add(layers.AvgPool2D(pool_size=(3, 3), strides=1, padding='valid'))
    model.add(lq.layers.QuantConv2D(64, (3, 3), padding='same', use_bias=False, **kwargs))
    model.add(layers.BatchNormalization(scale=False))
    model.add(layers.ReLU())

    # Layer 2
    model.add(layers.AvgPool2D(pool_size=(3, 3), strides=1, padding='valid'))
    model.add(lq.layers.QuantConv2D(64, (3, 3), padding='same', use_bias=False, **kwargs))
    model.add(layers.BatchNormalization(scale=False))
    model.add(layers.ReLU())

    # Layer 3
    model.add(lq.layers.QuantConv2D(64, (3, 3), padding='same', use_bias=False, **kwargs))
    model.add(layers.AvgPool2D(pool_size=(3, 3), strides=1, padding='valid'))
    model.add(layers.BatchNormalization(scale=False))
    model.add(layers.ReLU())

    # Layer 4
    model.add(lq.layers.QuantConv2D(64, (3, 3), padding='same', use_bias=False, **kwargs))
    model.add(layers.AvgPool2D(pool_size=(3, 3), strides=1, padding='valid'))
    model.add(layers.BatchNormalization(scale=False))
    model.add(layers.ReLU())

    # Layer 5
    model.add(layers.AvgPool2D(pool_size=(3, 3), strides=1, padding='valid'))
    model.add(lq.layers.QuantConv2D(22, (3, 3), padding='same', use_bias=False, **kwargs))
    model.add(layers.BatchNormalization(scale=False))
    model.add(layers.ReLU())

    model.add(layers.GlobalAveragePooling2D())

    model.add(lq.layers.QuantDense(128, use_bias=False, **kwargs))
    model.add(layers.BatchNormalization(scale=False))
    model.add(layers.ReLU())

    model.add(lq.layers.QuantDense(64, use_bias=False, **kwargs))
    model.add(layers.BatchNormalization(scale=False))
    model.add(layers.ReLU())

    model.add(lq.layers.QuantDense(32, use_bias=False, **kwargs))
    model.add(layers.BatchNormalization(scale=False))
    model.add(layers.ReLU())

    model.add(lq.layers.QuantDense(10, use_bias=False, **kwargs))
    model.add(layers.BatchNormalization(scale=False))
    model.add(layers.Activation("softmax"))

    return model

quantized_model = create_larq_model()


In [12]:
num_classes = 10

(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.cifar10.load_data()

train_images = train_images.reshape((50000, 32, 32, 3)).astype("float32")
test_images = test_images.reshape((10000, 32, 32, 3)).astype("float32")

# Normalize pixel values to be between -1 and 1
train_images, test_images = train_images / 127.5 - 1, test_images / 127.5 - 1

train_labels = tf.keras.utils.to_categorical(train_labels, num_classes)
test_labels = tf.keras.utils.to_categorical(test_labels, num_classes)

In [ ]:
quantized_model.compile(
    tf.keras.optimizers.Adam(lr=0.01, decay=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
trained_model = quantized_model.fit(
    train_images, 
    train_labels,
    batch_size=32, 
    epochs=100,
    validation_data=(test_images, test_labels),
    shuffle=True)


Epoch 1/100
1563/1563 [==============================] - 220s 141ms/step - loss: 2.1069 - accuracy: 0.1909 - val_loss: 2.3221 - val_accuracy: 0.1733
Epoch 2/100
1563/1563 [==============================] - 229s 146ms/step - loss: 1.9402 - accuracy: 0.2556 - val_loss: 1.9751 - val_accuracy: 0.2160
Epoch 3/100
1563/1563 [==============================] - 233s 149ms/step - loss: 1.9194 - accuracy: 0.2652 - val_loss: 2.2010 - val_accuracy: 0.1809
Epoch 4/100
1563/1563 [==============================] - 230s 147ms/step - loss: 1.8956 - accuracy: 0.2831 - val_loss: 2.1893 - val_accuracy: 0.1830
Epoch 5/100
1563/1563 [==============================] - 235s 150ms/step - loss: 1.8726 - accuracy: 0.3024 - val_loss: 2.1347 - val_accuracy: 0.2239
Epoch 6/100
1563/1563 [==============================] - 240s 154ms/step - loss: 1.8631 - accuracy: 0.3064 - val_loss: 2.1799 - val_accuracy: 0.1867
Epoch 7/100
1563/1563 [==============================] - 232s 149ms/step - loss: 1.8797 - accuracy: 0.2928